# Fast five-way golf-club CNN

Train a compact CNN to classify a club-head image as Driver, Wood, Hybrid, Iron, or Wedge. The notebook keeps related images from the same product together, applies identical resize and normalization to every split, and applies augmentation only to training images.

Run the cells from top to bottom. The best checkpoint is saved to `models/trained/club_type_5way_cnn.pt`. The reference set is useful for validating the workflow, but add varied club-head photos before relying on the model in production.

In [5]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
import random
import re
import sys
from time import perf_counter

import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "requirements.txt").is_file()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from swingsight.club_cnn import CHECKPOINT_FORMAT, DEFAULT_MEAN, DEFAULT_STD

CLASS_NAMES = ("driver", "wood", "hybrid", "iron", "wedge")
CLASS_TO_INDEX = {class_name: index for index, class_name in enumerate(CLASS_NAMES)}
DISPLAY_NAMES = {
    "driver": "Driver",
    "wood": "Fairway Wood",
    "hybrid": "Hybrid",
    "iron": "Iron",
    "wedge": "Wedge",
}
SOURCE_DIRS = {
    "driver": PROJECT_ROOT / "assets" / "club-reference-images" / "wood" / "driver",
    "wood": PROJECT_ROOT / "assets" / "club-reference-images" / "wood" / "wood",
    "hybrid": PROJECT_ROOT / "assets" / "club-reference-images" / "wood" / "hybrid",
    "iron": PROJECT_ROOT / "assets" / "club-reference-images" / "iron" / "irons",
    "wedge": PROJECT_ROOT / "assets" / "club-reference-images" / "iron" / "wedges",
}
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "trained" / "club_type_5way.pt"
INPUT_SIZE = 224
SEED = 42
BATCH_SIZE = 96
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}
ARCHITECTURE = "mobilenet_v3_small_v1"

try:
    from IPython import get_ipython

    RUNNING_IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    RUNNING_IN_NOTEBOOK = False

# Windows workers can hang while spawning notebook-defined dataset classes.
# Keep notebook runs reliable; use two workers when this code is moved to a script.
NUM_WORKERS = 0 if RUNNING_IN_NOTEBOOK else 2
PREFETCH_FACTOR = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Project: {PROJECT_ROOT}")
print(f"Classes: {', '.join(DISPLAY_NAMES[name] for name in CLASS_NAMES)}")
print(f"Input: {INPUT_SIZE}x{INPUT_SIZE} | batch size: {BATCH_SIZE}")
print(f"DataLoader workers: {NUM_WORKERS} | prefetch factor: {PREFETCH_FACTOR}")
if RUNNING_IN_NOTEBOOK:
    print("Using single-process loading for Windows/Jupyter reliability.")

Project: C:\Users\Matt Shaver\OneDrive\Desktop\SwingSight-AI
Classes: Driver, Fairway Wood, Hybrid, Iron, Wedge
Input: 224x224 | batch size: 96
DataLoader workers: 0 | prefetch factor: 2
Using single-process loading for Windows/Jupyter reliability.


## 1. Discover images and create a grouped split

The split is performed by product group, not by individual file. This prevents related views or downloaded variants of one club from appearing in both training and validation/test data.

In [6]:
@dataclass(frozen=True)
class Sample:
    path: Path
    label: int
    group: str


AUGMENTATION_SUFFIX = re.compile(
    r"(?:[_-](?:aug|augmentation|image|img|zoom|flip|flipped|rot|rotate|crop|perspective)\d*)+$",
    re.IGNORECASE,
)


def product_group(path: Path) -> str:
    """Collapse generated filename suffixes to the original image stem."""
    stem = path.stem.lower()
    previous = None
    while stem != previous:
        previous = stem
        stem = AUGMENTATION_SUFFIX.sub("", stem)
    return stem or path.stem.lower()


assert product_group(Path("driver_001.jpg")) == "driver_001"
assert product_group(Path("driver_001_aug1.jpg")) == "driver_001"
assert product_group(Path("driver_001_aug2.jpg")) == "driver_001"

all_samples: list[Sample] = []
for label, class_name in enumerate(CLASS_NAMES):
    directory = SOURCE_DIRS[class_name]
    images = sorted(
        path
        for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if len(images) < 4:
        raise ValueError(
            f"{class_name} needs at least four images; found {len(images)} in {directory}"
        )
    all_samples.extend(Sample(path, label, product_group(path)) for path in images)


def grouped_split(label: int) -> tuple[list[Sample], list[Sample], list[Sample]]:
    """Create the same per-class 60/30/10 split without crossing source groups."""
    grouped: dict[str, list[Sample]] = defaultdict(list)
    for sample in all_samples:
        if sample.label == label:
            grouped[sample.group].append(sample)

    groups = sorted(grouped)
    if len(groups) < 3:
        raise ValueError(
            f"{CLASS_NAMES[label]} needs at least three product groups; found {len(groups)}"
        )

    random.Random(SEED + label).shuffle(groups)
    test_count = max(1, round(len(groups) * 0.10))
    validation_count = max(1, round(len(groups) * 0.30))
    while test_count + validation_count >= len(groups):
        if validation_count > 1:
            validation_count -= 1
        else:
            test_count -= 1

    def flatten(group_names: list[str]) -> list[Sample]:
        return [sample for name in group_names for sample in grouped[name]]

    test_groups = groups[:test_count]
    validation_groups = groups[test_count:test_count + validation_count]
    train_groups = groups[test_count + validation_count:]
    return flatten(train_groups), flatten(validation_groups), flatten(test_groups)


train_samples, validation_samples, test_samples = [], [], []
for label in range(len(CLASS_NAMES)):
    train_part, validation_part, test_part = grouped_split(label)
    train_samples.extend(train_part)
    validation_samples.extend(validation_part)
    test_samples.extend(test_part)


def group_names(samples: list[Sample]) -> set[str]:
    return {f"{sample.label}:{sample.group}" for sample in samples}


train_groups = group_names(train_samples)
validation_groups = group_names(validation_samples)
test_groups = group_names(test_samples)
assert not train_groups & validation_groups
assert not train_groups & test_groups
assert not validation_groups & test_groups

print("class              train  validation  test")
for label, class_name in enumerate(CLASS_NAMES):
    counts = {
        split_name: sum(sample.label == label for sample in samples)
        for split_name, samples in (
            ("train", train_samples),
            ("validation", validation_samples),
            ("test", test_samples),
        )
    }
    print(
        f"{DISPLAY_NAMES[class_name]:<18}"
        f"{counts['train']:>5}{counts['validation']:>12}{counts['test']:>6}"
    )
print(
    f"Total: {len(all_samples)} images | "
    f"train={len(train_samples)} validation={len(validation_samples)} test={len(test_samples)}"
)

class              train  validation  test
Driver              300         150    50
Fairway Wood        300         150    50
Hybrid              300         150    50
Iron                300         150    50
Wedge               300         150    50
Total: 2500 images | train=1500 validation=750 test=250


## 2. Define preprocessing and data loaders

Every image is converted to RGB, resized to the model input size, converted to a tensor, and normalized with the checkpoint values. Training gets mild image augmentation before this shared deterministic preprocessing; validation and test do not.

In [7]:
IMAGENET_FILL = tuple(round(value * 255) for value in DEFAULT_MEAN)

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            INPUT_SIZE,
            scale=(0.82, 1.0),
            ratio=(0.90, 1.10),
            antialias=True,
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(
            degrees=8,
            translate=(0.06, 0.06),
            scale=(0.92, 1.05),
            shear=3,
            fill=IMAGENET_FILL,
        ),
        transforms.RandomPerspective(
            distortion_scale=0.12,
            p=0.25,
            fill=IMAGENET_FILL,
        ),
        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
        ),
        transforms.ToTensor(),
        transforms.Normalize(DEFAULT_MEAN, DEFAULT_STD),
    ]
)
validation_transform = transforms.Compose(
    [
        transforms.Resize((INPUT_SIZE, INPUT_SIZE), antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(DEFAULT_MEAN, DEFAULT_STD),
    ]
)


class ClubDataset(Dataset):
    def __init__(self, samples: list[Sample], transform: transforms.Compose) -> None:
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self.samples[index]
        with Image.open(sample.path) as opened:
            image = opened.convert("RGB")
        return self.transform(image), torch.tensor(sample.label, dtype=torch.long)


train_dataset = ClubDataset(train_samples, train_transform)
validation_dataset = ClubDataset(validation_samples, validation_transform)
test_dataset = ClubDataset(test_samples, validation_transform)


def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    loader_options = {
        "batch_size": BATCH_SIZE,
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": NUM_WORKERS > 0,
    }
    if NUM_WORKERS > 0:
        loader_options["prefetch_factor"] = PREFETCH_FACTOR
    return DataLoader(dataset, **loader_options)


train_loader = make_loader(train_dataset, shuffle=True)
validation_loader = make_loader(validation_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Augmentation fill color: {IMAGENET_FILL} (not white)")

Training samples: 1500
Validation samples: 750
Test samples: 250
Augmentation fill color: (124, 116, 104) (not white)


## 3. Verify preprocessing and labels

This quick check catches the two common pipeline failures before a CUDA run: inconsistent tensor shapes or normalization, and mismatched class labels between loaders.

In [8]:
train_batch, train_labels = next(iter(train_loader))
validation_batch, validation_labels = next(iter(validation_loader))

assert tuple(CLASS_TO_INDEX) == CLASS_NAMES
assert train_batch.shape[1:] == (3, INPUT_SIZE, INPUT_SIZE)
assert validation_batch.shape[1:] == (3, INPUT_SIZE, INPUT_SIZE)
assert train_labels.min().item() >= 0
assert train_labels.max().item() < len(CLASS_NAMES)
assert validation_labels.min().item() >= 0
assert validation_labels.max().item() < len(CLASS_NAMES)

train_counts = Counter(sample.label for sample in train_samples)
validation_counts = Counter(sample.label for sample in validation_samples)
test_counts = Counter(sample.label for sample in test_samples)
assert set(train_counts) == set(validation_counts) == set(test_counts) == set(range(len(CLASS_NAMES)))

print(f"Train batch: {tuple(train_batch.shape)}")
print(f"Validation batch: {tuple(validation_batch.shape)}")
print(f"Train tensor range: {train_batch.min().item():.2f} to {train_batch.max().item():.2f}")
print(f"Validation tensor range: {validation_batch.min().item():.2f} to {validation_batch.max().item():.2f}")
print("Shared class mapping:")
for class_name, label in CLASS_TO_INDEX.items():
    print(f"  {label}: {DISPLAY_NAMES[class_name]} ({class_name})")
print("Per-class image counts:")
for label, class_name in enumerate(CLASS_NAMES):
    print(
        f"  {DISPLAY_NAMES[class_name]:<14}"
        f"train={train_counts[label]:>4} "
        f"validation={validation_counts[label]:>4} "
        f"test={test_counts[label]:>4}"
    )

Train batch: (96, 3, 224, 224)
Validation batch: (96, 3, 224, 224)
Train tensor range: -2.12 to 2.64
Validation tensor range: -2.12 to 2.64
Shared class mapping:
  0: Driver (driver)
  1: Fairway Wood (wood)
  2: Hybrid (hybrid)
  3: Iron (iron)
  4: Wedge (wedge)
Per-class image counts:
  Driver        train= 300 validation= 150 test=  50
  Fairway Wood  train= 300 validation= 150 test=  50
  Hybrid        train= 300 validation= 150 test=  50
  Iron          train= 300 validation= 150 test=  50
  Wedge         train= 300 validation= 150 test=  50


## 4. Configure CUDA training

The lower learning rate gives validation metrics room to settle on this small dataset. Class weighting handles the modest class-count differences without changing the label mapping.

In [11]:
HEAD_EPOCHS = 5
FINE_TUNE_EPOCHS = 30
EPOCHS = HEAD_EPOCHS + FINE_TUNE_EPOCHS
LEARNING_RATE_HEAD = 1e-3
LEARNING_RATE_FINE_TUNE = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
LABEL_SMOOTHING_OPTIONS = (0.00, 0.05, 0.10)
PATIENCE = 7
GRADIENT_CLIP_NORM = 1.0
USE_CLASS_WEIGHTS = False
REQUIRE_CUDA = True
USE_AMP = True
PROFILE_BATCH_TIMING = False


def select_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda}")
        torch.backends.cudnn.benchmark = True
        return device
    if REQUIRE_CUDA:
        raise RuntimeError(
            "CUDA is required for this notebook, but this Python environment has no CUDA-enabled PyTorch build. "
            "Install the CUDA PyTorch wheel, restart the kernel, and run this cell again."
        )
    mps = getattr(torch.backends, "mps", None)
    return torch.device("mps") if mps is not None and mps.is_available() else torch.device("cpu")


def build_model(pretrained: bool) -> nn.Module:
    weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
    network = mobilenet_v3_small(weights=weights)
    network.classifier[-1] = nn.Linear(network.classifier[-1].in_features, len(CLASS_NAMES))
    return network


def set_trainable_layers(network: nn.Module, fine_tune: bool) -> None:
    for parameter in network.features.parameters():
        parameter.requires_grad = False
    for parameter in network.classifier.parameters():
        parameter.requires_grad = True
    if fine_tune:
        for block in network.features[-3:]:
            for parameter in block.parameters():
                parameter.requires_grad = True


def make_optimizer(network: nn.Module, learning_rate: float) -> torch.optim.Optimizer:
    return torch.optim.AdamW(
        (parameter for parameter in network.parameters() if parameter.requires_grad),
        lr=learning_rate,
        weight_decay=WEIGHT_DECAY,
    )


def autocast_context(device: torch.device):
    return torch.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=USE_AMP and device.type == "cuda",
    )


def evaluate(
    network: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    collect_predictions: bool = False,
) -> dict:
    network.eval()
    total_loss = torch.zeros((), device=device)
    total = 0
    top1_correct = torch.zeros((), device=device, dtype=torch.long)
    top2_correct = torch.zeros((), device=device, dtype=torch.long)
    actual, predicted, confidences, probabilities = [], [], [], []

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast_context(device):
                logits = network(images)
                loss = criterion(logits, labels)
            total_loss += loss.detach() * labels.size(0)
            total += labels.size(0)
            top1_correct += (logits.argmax(dim=1) == labels).sum()
            top2_correct += (
                logits.topk(k=min(2, len(CLASS_NAMES)), dim=1).indices == labels.unsqueeze(1)
            ).any(dim=1).sum()

            if collect_predictions:
                batch_probabilities = torch.softmax(logits.float(), dim=1)
                batch_confidences, batch_predictions = batch_probabilities.max(dim=1)
                actual.extend(labels.cpu().tolist())
                predicted.extend(batch_predictions.cpu().tolist())
                confidences.extend(batch_confidences.cpu().tolist())
                probabilities.extend(batch_probabilities.cpu().tolist())

    return {
        "loss": float((total_loss / max(1, total)).item()),
        "top1": float((top1_correct.float() / max(1, total)).item()),
        "top2": float((top2_correct.float() / max(1, total)).item()),
        "actual": actual,
        "predicted": predicted,
        "confidences": confidences,
        "probabilities": probabilities,
    }


def checkpoint_payload(network: nn.Module, validation: dict, epoch: int) -> dict:
    return {
        "format": CHECKPOINT_FORMAT,
        "task": "club_type_5way",
        "architecture": ARCHITECTURE,
        "class_names": list(CLASS_NAMES),
        "input_size": INPUT_SIZE,
        "mean": list(DEFAULT_MEAN),
        "std": list(DEFAULT_STD),
        "validation_loss": round(validation["loss"], 6),
        "validation_accuracy": round(validation["top1"], 6),
        "validation_top2_accuracy": round(validation["top2"], 6),
        "best_epoch": epoch,
        "label_smoothing": LABEL_SMOOTHING,
        "state_dict": {
            name: value.detach().cpu()
            for name, value in network.state_dict().items()
        },
    }


device = select_device()
model = build_model(pretrained=True).to(device)
set_trainable_layers(model, fine_tune=False)

if USE_CLASS_WEIGHTS:
    class_counts = Counter(sample.label for sample in train_samples)
    class_weights = torch.tensor(
        [len(train_samples) / (len(CLASS_NAMES) * class_counts[label]) for label in range(len(CLASS_NAMES))],
        dtype=torch.float32,
        device=device,
    )
else:
    class_weights = None

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING,
)
optimizer = make_optimizer(model, LEARNING_RATE_HEAD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.3,
    patience=2,
    min_lr=1e-6,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and device.type == "cuda")

print(f"Architecture: {ARCHITECTURE} | pretrained weights: yes")
print(f"Training {sum(parameter.numel() for parameter in model.parameters()):,} parameters on {device}.")
print(f"AMP enabled: {scaler.is_enabled()} | batch size: {BATCH_SIZE}")
print(f"Class weights: {'enabled' if USE_CLASS_WEIGHTS else 'disabled for balanced data'}")
print(f"Label smoothing candidates: {LABEL_SMOOTHING_OPTIONS} | selected: {LABEL_SMOOTHING}")

Using CUDA: NVIDIA GeForce RTX 4070 SUPER | CUDA 12.8
Architecture: mobilenet_v3_small_v1 | pretrained weights: yes
Training 1,522,981 parameters on cuda.
AMP enabled: True | batch size: 96
Class weights: disabled for balanced data
Label smoothing candidates: (0.0, 0.05, 0.1) | selected: 0.05


## 5. Train and save the best checkpoint

Validation is measured after every epoch using the deterministic validation loader. The checkpoint is selected by validation accuracy, while early stopping limits a run that stops improving.

In [12]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
best_validation_loss = float("inf")
best_validation_accuracy = 0.0
best_validation_accuracy_epoch = 0
stale_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    if epoch == HEAD_EPOCHS + 1:
        set_trainable_layers(model, fine_tune=True)
        optimizer = make_optimizer(model, LEARNING_RATE_FINE_TUNE)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.3,
            patience=2,
            min_lr=1e-6,
        )
        print(f"Unfroze the last three backbone blocks for fine-tuning at epoch {epoch}.")

    model.train()
    train_loss_total = torch.zeros((), device=device)
    train_correct = torch.zeros((), device=device, dtype=torch.long)
    train_count = 0
    data_time_total = 0.0
    compute_time_total = 0.0
    batch_iterator = iter(train_loader)

    while True:
        data_started = perf_counter()
        try:
            images, labels = next(batch_iterator)
        except StopIteration:
            break
        data_time_total += perf_counter() - data_started

        if PROFILE_BATCH_TIMING and device.type == "cuda":
            torch.cuda.synchronize()
        compute_started = perf_counter()
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        train_loss_total += loss.detach() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum()
        train_count += labels.size(0)
        if PROFILE_BATCH_TIMING and device.type == "cuda":
            torch.cuda.synchronize()
        compute_time_total += perf_counter() - compute_started

    validation = evaluate(model, validation_loader, criterion, device)
    scheduler.step(validation["loss"])
    train_loss = float((train_loss_total / max(1, train_count)).item())
    train_accuracy = float((train_correct.float() / max(1, train_count)).item())
    current_learning_rate = optimizer.param_groups[0]["lr"]
    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": validation["loss"],
        "validation_accuracy": validation["top1"],
        "validation_top2_accuracy": validation["top2"],
        "learning_rate": current_learning_rate,
    }
    if PROFILE_BATCH_TIMING:
        record["data_seconds"] = data_time_total
        record["compute_seconds"] = compute_time_total
    history.append(record)

    print(
        f"epoch {epoch:02d}/{EPOCHS} "
        f"train_loss={train_loss:.4f} "
        f"train_top1={train_accuracy:.3f} "
        f"validation_loss={validation['loss']:.4f} "
        f"validation_top1={validation['top1']:.3f} "
        f"validation_top2={validation['top2']:.3f} "
        f"lr={current_learning_rate:.2e}"
    )
    if PROFILE_BATCH_TIMING:
        print(
            f"  data_wait={data_time_total:.1f}s "
            f"compute={compute_time_total:.1f}s"
        )

    if validation["top1"] > best_validation_accuracy:
        best_validation_accuracy = validation["top1"]
        best_validation_accuracy_epoch = epoch

    if validation["loss"] < best_validation_loss:
        best_validation_loss = validation["loss"]
        stale_epochs = 0
        torch.save(checkpoint_payload(model, validation, epoch), CHECKPOINT_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without validation-loss improvement.")
            break

print(f"Best validation loss: {best_validation_loss:.4f}")
print(
    f"Best validation accuracy: {best_validation_accuracy:.3f} "
    f"at epoch {best_validation_accuracy_epoch}"
)
print(f"Saved checkpoint: {CHECKPOINT_PATH}")

epoch 01/35 train_loss=1.1824 train_top1=0.533 validation_loss=1.4747 validation_top1=0.363 validation_top2=0.692 lr=1.00e-03
epoch 02/35 train_loss=0.8660 train_top1=0.708 validation_loss=1.0288 validation_top1=0.583 validation_top2=0.841 lr=1.00e-03
epoch 03/35 train_loss=0.7206 train_top1=0.783 validation_loss=0.9372 validation_top1=0.628 validation_top2=0.893 lr=1.00e-03
epoch 04/35 train_loss=0.6800 train_top1=0.794 validation_loss=0.8757 validation_top1=0.693 validation_top2=0.908 lr=1.00e-03
epoch 05/35 train_loss=0.5974 train_top1=0.852 validation_loss=0.8211 validation_top1=0.713 validation_top2=0.919 lr=1.00e-03
Unfroze the last three backbone blocks for fine-tuning at epoch 6.
epoch 06/35 train_loss=0.5487 train_top1=0.876 validation_loss=0.6205 validation_top1=0.821 validation_top2=0.960 lr=2.00e-04
epoch 07/35 train_loss=0.4631 train_top1=0.921 validation_loss=0.6091 validation_top1=0.820 validation_top2=0.968 lr=2.00e-04
epoch 08/35 train_loss=0.4247 train_top1=0.941 vali

## 6. Diagnose the input pipeline

PIL decoding and torchvision augmentation run on CPU. This optional profile separates time spent waiting for the next batch from time spent executing a GPU forward pass. Set `PROFILE_BATCH_TIMING = True` before training for per-epoch timing, or run this cell after training for a short loader-only check.

In [13]:
def profile_input_pipeline(loader: DataLoader, batches: int = 20) -> None:
    iterator = iter(loader)
    data_seconds = 0.0
    gpu_seconds = 0.0

    for _ in range(batches):
        started = perf_counter()
        try:
            images, labels = next(iterator)
        except StopIteration:
            break
        data_seconds += perf_counter() - started

        if device.type == "cuda":
            torch.cuda.synchronize()
        started = perf_counter()
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.inference_mode(), autocast_context(device):
            _ = model(images)
        if device.type == "cuda":
            torch.cuda.synchronize()
        gpu_seconds += perf_counter() - started

    print(f"Profiled batches: {batches}")
    print(f"Average batch loading and CPU preprocessing: {data_seconds / max(1, batches):.4f}s")
    print(f"Average transfer and GPU forward pass: {gpu_seconds / max(1, batches):.4f}s")
    print("PIL decoding and torchvision augmentation remain CPU work; increase workers if data time dominates.")


profile_input_pipeline(train_loader)

Profiled batches: 20
Average batch loading and CPU preprocessing: 0.2486s
Average transfer and GPU forward pass: 0.0337s
PIL decoding and torchvision augmentation remain CPU work; increase workers if data time dominates.


## 7. Monitor CUDA utilization

In a PowerShell terminal, run `nvidia-smi -l 1` while training. In Windows Task Manager, choose the GPU's **CUDA** or **Compute** graph rather than relying only on the **3D** graph. If data-wait time is much larger than GPU time, PIL decoding and CPU augmentation are limiting throughput; increase workers carefully or reduce augmentation complexity.

## 7. Evaluate the held-out test set

This cell reports top-1 and top-2 accuracy, the confusion matrix, per-class precision and recall, the most common confusions, and confidence-threshold coverage. It then reuses the loaded model for local inference timing instead of loading checkpoint weights for every image.

In [14]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model = build_model(pretrained=False).to(device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

validation = evaluate(model, validation_loader, criterion, device, collect_predictions=True)
test = evaluate(model, test_loader, criterion, device, collect_predictions=True)
print(f"Best checkpoint epoch: {checkpoint.get('best_epoch', 'unknown')}")
print(f"Validation loss: {validation['loss']:.4f}")
print(f"Validation top-1 accuracy: {validation['top1']:.3f}")
print(f"Validation top-2 accuracy: {validation['top2']:.3f}")
print(f"Held-out test loss: {test['loss']:.4f}")
print(f"Held-out test top-1 accuracy: {test['top1']:.3f}")
print(f"Held-out test top-2 accuracy: {test['top2']:.3f}")

confusion_matrix = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)
for actual, predicted in zip(test["actual"], test["predicted"]):
    confusion_matrix[actual, predicted] += 1

print("\nConfusion matrix (rows=actual, columns=predicted):")
print("                  " + " ".join(f"{DISPLAY_NAMES[name][:8]:>10}" for name in CLASS_NAMES))
for index, class_name in enumerate(CLASS_NAMES):
    values = " ".join(f"{value:>10}" for value in confusion_matrix[index])
    print(f"{DISPLAY_NAMES[class_name]:<18} {values}")

print("\nPer-class precision / recall:")
for index, class_name in enumerate(CLASS_NAMES):
    true_positive = confusion_matrix[index, index]
    precision = true_positive / max(1, confusion_matrix[:, index].sum())
    recall = true_positive / max(1, confusion_matrix[index, :].sum())
    print(f"{DISPLAY_NAMES[class_name]:<14} precision={precision:.3f} recall={recall:.3f}")

confusions = []
for actual in range(len(CLASS_NAMES)):
    for predicted in range(len(CLASS_NAMES)):
        if actual != predicted and confusion_matrix[actual, predicted] > 0:
            confusions.append(
                (
                    int(confusion_matrix[actual, predicted]),
                    DISPLAY_NAMES[CLASS_NAMES[actual]],
                    DISPLAY_NAMES[CLASS_NAMES[predicted]],
                )
            )
print("\nMost common class confusions:")
for count, actual_name, predicted_name in sorted(confusions, reverse=True)[:10]:
    print(f"  {actual_name} -> {predicted_name}: {count}")

CONFIDENCE_THRESHOLD = 0.60
known = [confidence >= CONFIDENCE_THRESHOLD for confidence in test["confidences"]]
correct = [actual == predicted for actual, predicted in zip(test["actual"], test["predicted"])]
known_count = sum(known)
known_correct = sum(is_known and is_correct for is_known, is_correct in zip(known, correct))
unknown_count = len(known) - known_count
coverage = known_count / max(1, len(known))
covered_accuracy = known_correct / max(1, known_count)
print(f"\nConfidence threshold: {CONFIDENCE_THRESHOLD:.2f}")
print(f"Known predictions: {known_count}/{len(known)}")
print(f"Unknown predictions: {unknown_count}")
print(f"Coverage: {coverage:.3f}")
print(f"Accuracy on covered predictions: {covered_accuracy:.3f}")
print("Class-specific thresholds: not applied; validation data does not justify changing the global threshold yet.")

checkpoint["test_accuracy"] = round(float(test["top1"]), 6)
checkpoint["test_top2_accuracy"] = round(float(test["top2"]), 6)
checkpoint["confidence_threshold"] = CONFIDENCE_THRESHOLD
torch.save(checkpoint, CHECKPOINT_PATH)


inference_model = model
inference_model.eval()


@torch.inference_mode()
def predict_club_type(
    image_path: str | Path,
    minimum_confidence: float = CONFIDENCE_THRESHOLD,
) -> dict:
    with Image.open(image_path) as opened:
        image = validation_transform(opened.convert("RGB")).unsqueeze(0).to(
            device,
            non_blocking=True,
        )
    with autocast_context(device):
        probabilities = torch.softmax(inference_model(image).float(), dim=1)[0]
    confidence, prediction = probabilities.max(dim=0)
    confidence_value = float(confidence.item())
    label = CLASS_NAMES[int(prediction.item())]
    if confidence_value < minimum_confidence:
        return {
            "label": "Unknown",
            "confidence": round(confidence_value, 3),
            "probabilities": {
                DISPLAY_NAMES[name]: round(float(score), 6)
                for name, score in zip(CLASS_NAMES, probabilities.tolist())
            },
            "reasoning": (
                f"club_type_5way CNN confidence {confidence_value:.2f} is below "
                f"the required {minimum_confidence:.2f}."
            ),
        }
    return {
        "label": DISPLAY_NAMES[label],
        "confidence": round(confidence_value, 3),
        "probabilities": {
            DISPLAY_NAMES[name]: round(float(score), 6)
            for name, score in zip(CLASS_NAMES, probabilities.tolist())
        },
        "reasoning": f"club_type_5way CNN predicted {DISPLAY_NAMES[label]}.",
    }


example_path = test_samples[0].path
print(f"\nExample: {example_path.name}")
print(predict_club_type(example_path))

for _ in range(3):
    predict_club_type(example_path, minimum_confidence=0.0)
if device.type == "cuda":
    torch.cuda.synchronize()
runs, started = 30, perf_counter()
for _ in range(runs):
    predict_club_type(example_path, minimum_confidence=0.0)
if device.type == "cuda":
    torch.cuda.synchronize()
print(f"Average local inference time: {(perf_counter() - started) * 1000 / runs:.1f} ms/image")
print("Inference model is loaded once and reused for all predictions.")
print(f"Deployable checkpoint: {CHECKPOINT_PATH}")

Best checkpoint epoch: 31
Validation loss: 0.2852
Validation top-1 accuracy: 0.988
Validation top-2 accuracy: 1.000
Held-out test loss: 0.2812
Held-out test top-1 accuracy: 0.992
Held-out test top-2 accuracy: 1.000

Confusion matrix (rows=actual, columns=predicted):
                      Driver   Fairway      Hybrid       Iron      Wedge
Driver                     50          0          0          0          0
Fairway Wood                1         48          1          0          0
Hybrid                      0          0         50          0          0
Iron                        0          0          0         50          0
Wedge                       0          0          0          0         50

Per-class precision / recall:
Driver         precision=0.980 recall=1.000
Fairway Wood   precision=1.000 recall=0.960
Hybrid         precision=0.980 recall=1.000
Iron           precision=1.000 recall=1.000
Wedge          precision=1.000 recall=1.000

Most common class confusions:
  Fairwa